In [1]:
import pandas as pd
from pathlib import Path
import warnings
import os
import joblib


from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation Metrics
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
DATA_PATH = Path('../data/student_dropout_dataset_v3.csv')
df = pd.read_csv(DATA_PATH)

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


In [2]:
# 1. Parental Education
df['Parental_Education'] = df['Parental_Education'].fillna('Unknown')
education_mapping = {
    'Unknown': 0,
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}
df['Parental_Education'] = df['Parental_Education'].replace(education_mapping)

binary_columns = ['Internet_Access', 'Part_Time_Job', 'Scholarship']
for column in binary_columns:
    df[column] = (df[column] == 'Yes').astype(int)

gender_mapping = {
    'Female': 1,
    'Male': 0}
df['Gender'] = df['Gender'].replace(gender_mapping)

semester_mapping = {
    'Year 1': 1,
    'Year 2': 2,
    'Year 3': 3,
    'Year 4': 4
}
df['Semester'] = df['Semester'].replace(semester_mapping)

# GPA_trend se NEPOČÍTÁ zde — přesunuto dovnitř HolmanImpute.transform(),
# aby byl preprocessor self-contained a fungoval i na nových datech bez ručního pre-processingu.


In [3]:
y = df['Dropout']
X = df.drop(columns=['Dropout'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
class HolmanImpute(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stats = {}

    def fit(self, X, y=None):
        self.stats['income_medians'] = X.groupby('Parental_Education')['Family_Income'].median()
        self.stats['study_means'] = X.groupby('Parental_Education')['Study_Hours_per_Day'].mean()
        self.stats['stress_median'] = X['Stress_Index'].median()
        self.stats['age_mean'] = X['Age'].mean()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy['Family_Income'] = X_copy['Family_Income'].fillna(X_copy['Parental_Education'].map(self.stats['income_medians']))
        X_copy['Study_Hours_per_Day'] = X_copy['Study_Hours_per_Day'].fillna(X_copy['Parental_Education'].map(self.stats['study_means']))
        X_copy['Stress_Index'] = X_copy['Stress_Index'].fillna(self.stats['stress_median'])
        X_copy['Age_Gap'] = X_copy['Age'] - self.stats['age_mean']

        # GPA_trend: rozdíl mezi kumulativním GPA a GPA aktuálního semestru
        # Počítáme ZDE (uvnitř pipeline), ne před splitem — pipeline je tak self-contained
        # Kladná hodnota = student dříve dosahoval lepších výsledků (klesající trend)
        X_copy['GPA_trend'] = X_copy['CGPA'] - X_copy['Semester_GPA']

        # Smazání nepotřebných sloupců
        # Parental_Education bylo testováno jako příznak — změna ROC-AUC < 0.003, proto vyřazeno jako redundantní
        # GPA (první semestrální základ) a Semester_GPA již zachyceny přes CGPA a GPA_trend
        cols_to_drop = ['Student_ID', 'Age', 'Parental_Education', 'GPA', 'Semester_GPA']
        cols_to_drop = [c for c in cols_to_drop if c in X_copy.columns]
        X_copy = X_copy.drop(columns=cols_to_drop)
        return X_copy

# 2. Sestavení finální Roury
# První krok: HolmanImpute (imputace, feature engineering, drop zbytečných sloupců)
# Druhý krok: ColumnTransformer (přeškáluje čísla a zakóduje Fakultu, zbytek nechá být)
numeric_cols = ['Family_Income', 'Study_Hours_per_Day', 'Stress_Index', 'Age_Gap', 'CGPA', 'GPA_trend']

col_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Department'])
    ],
    remainder='passthrough',  # Internet_Access, Semester atd. projdou beze změny
    verbose_feature_names_out=False
)

# Finální Pipeline — self-contained, funguje na libovolných raw datech
preprocessor = Pipeline(steps=[
    ('imputer', HolmanImpute()),
    ('transformer', col_transformer)
])

# Výstup jako Pandas DataFrame (kvůli grafům a SHAP)
col_transformer.set_output(transform="pandas")

# 3. Aplikace a EXPORT
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f"Sloupce po předzpracování ({len(X_train_prep.columns)}):")
print(list(X_train_prep.columns))
print(f"\nX_train_prep: {X_train_prep.shape}")
print(f"X_test_prep:  {X_test_prep.shape}")

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump((X_train_prep, X_test_prep, y_train, y_test), '../data/processed/split_data.pkl')
print("\nPreprocessor a data uloženy.")


Sloupce po předzpracování (19):
['Family_Income', 'Study_Hours_per_Day', 'Stress_Index', 'Age_Gap', 'CGPA', 'GPA_trend', 'Department_Arts', 'Department_Business', 'Department_CS', 'Department_Engineering', 'Department_Science', 'Gender', 'Internet_Access', 'Attendance_Rate', 'Assignment_Delay_Days', 'Travel_Time_Minutes', 'Part_Time_Job', 'Scholarship', 'Semester']

X_train_prep: (8000, 19)
X_test_prep:  (2000, 19)

Preprocessor a data uloženy.
